<a href="https://colab.research.google.com/github/ssprajapati2021/Hybrid-RAG-Fine-Tuning/blob/main/notebook/Solution_V2_FineTuned_RAG_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

*****Setup to Get Files needed from previous Notebook*****

In [2]:
# Mount Google Drive to access the artifacts
from google.colab import drive
import os

drive.mount('/content/drive')

# Define Paths
artifact_path = "/content/drive/MyDrive/corporate_policies"

# Model Uses in previous notebooks

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [ ]:
# Load the base model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load LoRA Adapter
router_model = PeftModel.from_pretrained(
    base_model,
    os.path.join(
        artifact_path,
        "intent_lora_best"
    )
)

# Merge Adapter
router_model = router_model.merge_and_unload()

print("✅ LoRA merged successfully.")

In [3]:
# Building the Router Prompt

ROUTER_PROMPT = """
You are an intent classification assistant.

Your task is to identify the user's intent and category.

Return ONLY valid JSON.

Do not explain.
Do not include markdown.
Do not include extra text.

Example 1
User: Where is my order?
Output:
{"intent":"track_order","category":"ORDER"}

Example 2
User: I forgot my password.
Output:
{"intent":"recover_password","category":"ACCOUNT"}

Example 3
User: I want my money back.
Output:
{"intent":"get_refund","category":"REFUND"}

Example 4
User: Can I pay using PayPal?
Output:
{"intent":"check_payment_methods","category":"PAYMENT"}

User:
{query}

Output:
"""


In [ ]:
import json
import re
import torch

def extract_intent(query):

    prompt = ROUTER_PROMPT.format(query=query)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(router_model.device)

    outputs = router_model.generate(

        **inputs,

        max_new_tokens=30,

        do_sample=False
    )

    raw = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    match = re.search(r"\{.*?\}", raw, re.DOTALL)

    if match:

        try:

            return json.loads(match.group())

        except:

            pass

    return None

In [ ]:
# Intent to Search
intent_to_search = {

    "track_order":"# Track Order",

    "check_refund_policy":"# Refund Policy",

    "recover_password":"# Password Reset",

    "check_payment_methods":"# Payment Methods",

    "contact_customer_service":"# Contact Customer Service",

    "delivery_period":"# Shipping Delays",

    "newsletter_subscription":"# Subscription Cancellation"
}

In [ ]:
##########################
# Define Retrieval
###########################

def retrieve_context(query):

    prediction = extract_intent(query)

    if prediction:
        intent = prediction.get("intent")
        search_query = intent_to_search.get(intent, query)
    else:
        intent = None
        search_query = query

    docs = vector_db.similarity_search(search_query, k=1)

    context = docs[0].page_content if docs else ""

    return {
        "intent": intent,
        "search_query": search_query,
        "context": context
    }

##############################
#  Define Hybrid RAG
##############################

def generate_hybrid_rag(query):

    # Step 1: Intent Routing
    prediction = extract_intent(query)

    if prediction:
        intent = prediction.get("intent")
        category = prediction.get("category")
        search_query = intent_to_search.get(intent, query)
    else:
        intent = "Unknown"
        category = "Unknown"
        search_query = query

    # Step 2: Retrieve SOP
    docs = vector_db.similarity_search(search_query, k=1)

    context = docs[0].page_content if docs else ""

    # Step 3: Final RAG Prompt
    rag_prompt = f"""
You are a customer support assistant.

Answer ONLY using the retrieved SOP below.
If the SOP does not contain the answer, state that the information is unavailable.

Retrieved SOP:
{context}

Customer Question:
{query}

Answer:
"""

    # Step 4: Generate Final Response
    answer = generate_baseline(rag_prompt)

    # Step 5: Return full pipeline output
    return {
        "query": query,
        "intent": intent,
        "category": category,
        "search_query": search_query,
        "retrieved_context": context,
        "answer": answer
    }

In [ ]:
# Validate the complete Hybrid RAG pipeline

query = outputs["test_query"]

result = generate_hybrid_rag(query)

print(f"User Query:\n{result["query"]}")

print(f"\nPredicted Intent:\n{result["intent"]}")

print(f"\nCategory:\n{result["category"]}")

print(f"\nSearch Query:\n{result["search_query"]}")

print(f"\nRetrieved SOP (First 300 Characters):\n{result["retrieved_context"][:300]}")

print("\nFinal Answer:")
print(result["answer"])

### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [ ]:
def evaluate_router(df):

    results = []

    for _, row in df.iterrows():

        prediction = extract_intent(row["instruction"])

        valid_json = prediction is not None

        predicted_intent = (
            prediction.get("intent")
            if valid_json else None
        )

        predicted_category = (
            prediction.get("category")
            if valid_json else None
        )

        results.append({

            "query": row["instruction"],

            "ground_truth_intent": row["intent"],
            "predicted_intent": predicted_intent,

            "ground_truth_category": row["category"],
            "predicted_category": predicted_category,

            "valid_json": valid_json

        })

    return pd.DataFrame(results)

In [ ]:
from rapidfuzz import fuzz

# Load held-out test split
df_test = pd.read_csv(
    os.path.join(
        artifact_path,
        "df_test.csv"
    )
)

print(f"Test Samples: {len(df_test)}")

print(f"Print First 5 records from df_test:\n{df_test.head()}")

#############################
# Intent Router Running
#############################
router_results = evaluate_router(df_test)

router_results.head()


In [ ]:
# Format Adherence
format_adherence = (

    router_results["valid_json"].mean()

) * 100

print(f"Format Adherence: {format_adherence:.2f}%")

# Exact Match
exact_match = (

    (
        router_results["ground_truth_intent"]
        ==
        router_results["predicted_intent"]
    ).mean()

) * 100

print(f"Exact Match Accuracy: {exact_match:.2f}%")

# Fuzzy Match
threshold = 90

fuzzy_matches = []

for _, row in router_results.iterrows():

    if pd.isna(row["predicted_intent"]):

        fuzzy_matches.append(False)

        continue

    score = fuzz.ratio(

        row["ground_truth_intent"],
        row["predicted_intent"]

    )

    fuzzy_matches.append(

        score >= threshold

    )

fuzzy_match = (

    sum(fuzzy_matches)

    /

    len(fuzzy_matches)

) * 100

print(f"Fuzzy Match Accuracy: {fuzzy_match:.2f}%")

In [ ]:
# Summary Table
router_test_metrics = pd.DataFrame({

    "Metric": [

        "Format Adherence",

        "Exact Match",

        "Fuzzy Match"

    ],

    "Value (%)": [

        round(format_adherence, 2),

        round(exact_match, 2),

        round(fuzzy_match, 2)

    ]

})

router_test_metrics

##########################
# Saving Results
###########################
router_results.to_csv(

    os.path.join(

        artifact_path,

        "router_results.csv"

    ),

    index=False

)

router_test_metrics.to_csv(

    os.path.join(

        artifact_path,

        "router_test_metrics.csv"

    ),

    index=False

)

print("✅ Router evaluation results saved.")

#### Evaluate Intent Router on Adversarial Queries

To evaluate the robustness of the fine-tuned intent router, an adversarial subset is created from the held-out test split using regex filtering. Customer queries containing sentiment or hedging words such as **still**, **never**, **terrible**, and **frustrated** are selected. The same evaluation framework is then applied to measure Format Adherence, Exact Match, and Fuzzy Match under more challenging user inputs.

In [ ]:
# Evaluate Adversarial Subset
import re

pattern = r"\b(still|never|terrible|frustrated)\b"

adversarial_df = df_test[
    df_test["instruction"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )
].copy()

print(f"Adversarial Samples: {len(adversarial_df)}")

adversarial_df.head()


In [ ]:
adversarial_results = evaluate_router(adversarial_df)

format_adherence = (
    adversarial_results["valid_json"].mean()
) * 100

exact_match = (
    (
        adversarial_results["ground_truth_intent"]
        ==
        adversarial_results["predicted_intent"]
    ).mean()
) * 100

adversarial_metrics = pd.DataFrame({

    "Metric":[

        "Format Adherence",

        "Exact Match",

        "Fuzzy Match"

    ],

    "Value (%)":[

        round(format_adherence,2),

        round(exact_match,2),

        round(fuzzy_match,2)

    ]

})

adversarial_metrics

#### Evaluate Final Hybrid RAG Responses

The complete Hybrid RAG pipeline is evaluated on the held-out test split. For each customer query, the fine-tuned intent router predicts the intent, retrieves the relevant SOP using vector search, and generates the final response. The generated responses are compared with the SOP-grounded reference chunks using ROUGE-1, ROUGE-L, and BLEU to measure semantic similarity and answer quality.

In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

import numpy as np

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

smooth = SmoothingFunction().method1

hybrid_scores = []

for _, row in df_test.iterrows():

    query = row["instruction"]

    reference = get_chunk_reference(row)

    result = generate_hybrid_rag(query)

    answer = result["answer"]

    rouge = scorer.score(
        reference,
        answer
    )

    bleu = sentence_bleu(
        [reference.split()],
        answer.split(),
        smoothing_function=smooth
    )

    hybrid_scores.append({

        "rouge1": rouge["rouge1"].fmeasure,

        "rougeL": rouge["rougeL"].fmeasure,

        "bleu": bleu

    })

In [ ]:
   hybrid_metrics = pd.DataFrame({

    "Metric":[

        "ROUGE-1",

        "ROUGE-L",

        "BLEU"

    ],

    "Value":[

        np.mean([

            x["rouge1"]

            for x in hybrid_scores

        ]),

        np.mean([

            x["rougeL"]

            for x in hybrid_scores

        ]),

        np.mean([

            x["bleu"]

            for x in hybrid_scores

        ])

    ]

})

hybrid_metrics

print("Hybrid RAG")

print(
    "ROUGE-1:",
    hybrid_metrics.iloc[0]["Value"]
)

print(
    "ROUGE-L:",
    hybrid_metrics.iloc[1]["Value"]
)

print(
    "BLEU:",
    hybrid_metrics.iloc[2]["Value"]
)

#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [ ]:
# Load V1 Metrics
v1_metrics = pd.read_csv(
    os.path.join(
        artifact_path,
        "v1_metrics.csv"
    )
)

v1_metrics

#######################
# Compare V1 vs V2
#######################
v1_v2_comparison = v1_metrics.merge(
    hybrid_metrics,
    on="Metric",
    suffixes=("_V1", "_V2")
)

#######################
# Improvement %
#######################
v1_v2_comparison["Improvement (%)"] = (
    (
        v1_v2_comparison["Value_V2"] -
        v1_v2_comparison["Value_V1"]
    )
    /
    v1_v2_comparison["Value_V1"]
    * 100
).round(2)

v1_v2_comparison

#### Observation

Solution V2 (Hybrid RAG) is compared against Solution V1 (Naive RAG) to quantify the impact of fine-tuning. Since both systems use the same retrieval pipeline, any improvement in ROUGE and BLEU scores is attributed to the fine-tuned intent router, which improves retrieval relevance by predicting structured intents before vector search.

### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [ ]:
# Load previous V1 metrics
v1_metrics = pd.read_csv(
    os.path.join(
        artifact_path,
        "v1_metrics.csv"
    )
)


comparative_results = []

for _, row in df_test.iterrows():

    query = row["instruction"]

    reference = get_chunk_reference(row)

    baseline = generate_baseline(query)

    #naive_rag = generate_naive_rag(query)

    hybrid = generate_hybrid_rag(query)["answer"]

    comparative_results.append({

        "query": query,

        "intent": row["intent"],

        "reference": reference,

        "baseline_output": baseline,

        "naive_rag_output": naive_rag,

        "hybrid_rag_output": hybrid

    })

comparative_results = pd.DataFrame(comparative_results)

comparative_results.head()

In [ ]:
comparative_summary = pd.DataFrame({

    "Metric": [
        "ROUGE-1",
        "ROUGE-L",
        "BLEU"
    ],

    "Baseline": [
        baseline_metrics.loc[
            baseline_metrics["Metric"]=="ROUGE-1",
            "Value"
        ].values[0],

        baseline_metrics.loc[
            baseline_metrics["Metric"]=="ROUGE-L",
            "Value"
        ].values[0],

        baseline_metrics.loc[
            baseline_metrics["Metric"]=="BLEU",
            "Value"
        ].values[0]
    ],

    "Naive RAG": [
        v1_metrics.loc[
            v1_metrics["Metric"]=="ROUGE-1",
            "Value"
        ].values[0],

        v1_metrics.loc[
            v1_metrics["Metric"]=="ROUGE-L",
            "Value"
        ].values[0],

        v1_metrics.loc[
            v1_metrics["Metric"]=="BLEU",
            "Value"
        ].values[0]
    ],

    "Hybrid RAG": [
        hybrid_metrics.loc[
            hybrid_metrics["Metric"]=="ROUGE-1",
            "Value"
        ].values[0],

        hybrid_metrics.loc[
            hybrid_metrics["Metric"]=="ROUGE-L",
            "Value"
        ].values[0],

        hybrid_metrics.loc[
            hybrid_metrics["Metric"]=="BLEU",
            "Value"
        ].values[0]
    ]

})

comparative_summary

In [ ]:
comparative_results.to_csv(
    os.path.join(
        artifact_path,
        "Comparative_Results_Full.csv"
    ),
    index=False
)

comparative_summary.to_csv(
    os.path.join(
        artifact_path,
        "Comparative_Results_Summary.csv"
    ),
    index=False
)

print("✅ Comparative analysis exported successfully.")

---
## END-OF-NOTEBOOK CHECKLIST (FINAL)

> **IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [ ] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [ ] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [ ] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [ ] **4.5** All 3 architectures scored with SOP-grounded references
- [ ] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [ ] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**